# Mirror 7 — GPU Training

This notebook trains the optional learned language/response realization layer around the existing Mirror 7 architecture.

The trained component does not replace the Mirror 7 cognitive runtime.

In [ ]:
import subprocess, sys, pathlib
REPO = 'https://github.com/realahmad07/Mirror-7.git'
ROOT = pathlib.Path('/content/Mirror-7')
if not ROOT.exists():
    subprocess.run(['git','clone',REPO,str(ROOT)], check=True)
%cd /content/Mirror-7
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-training.txt'], check=True)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Connect a GPU runtime before training.')


In [ ]:
from pathlib import Path
data = Path('training/data/smoke.jsonl')
if not data.exists():
    subprocess.run([sys.executable,'-m','training.make_smoke_dataset'], check=True)
subprocess.run([sys.executable,'-m','training.validate_dataset','--dataset',str(data)], check=True)


## Replace the smoke dataset before the real run

Set DATASET to the validated JSONL containing the actual Mirror 7 training examples. Keep train, validation, and test records separated.

In [ ]:
DATASET = 'training/data/smoke.jsonl'  # replace with the real dataset
OUTPUT = 'artifacts/mirror7_response.pt'
EPOCHS = 5
BATCH_SIZE = 8
subprocess.run([
    sys.executable,'-m','training.train',
    '--dataset',DATASET,
    '--output',OUTPUT,
    '--epochs',str(EPOCHS),
    '--batch-size',str(BATCH_SIZE),
], check=True)


In [ ]:
subprocess.run([
    sys.executable,'-m','training.evaluate',
    '--dataset',DATASET,
    '--checkpoint',OUTPUT,
    '--split','test',
    '--temperature','0',
], check=True)


## Promotion rule

Do not connect the checkpoint to the public workspace merely because training completes. First inspect held-out outputs, preserve the dataset fingerprint, run the full backend/runtime regression suite, and only then promote the learned component.